# InfiniteTalk｜G4高速版 v9｜最初に画像・音声をアップロード｜生成時間付きZIPダウンロード

**入力ファイルのアップロードを最初に行い、その後で環境構築・モデル取得を開始するColabノートブックです。** 画像1枚＋音声1つで、音声の実際の長さに合わせたアバター動画を作成します（40秒固定ではありません）。

### v9の追加点
- **①で最初に画像と音声をアップロード**。時間のかかるモデルのダウンロードは⑥。実行を続ける間に入力を後から用意する必要はありません
- 生成完了後に **MP4＋`generation_time_report.txt`＋`generation_time_report.json`＋計測CSV** をZIPへまとめ、ブラウザに一括ダウンロードをリクエスト
- ⑧でモデル読み込み・推論の所要時間、⑨でMP4仕上げ時間を自動記録。区間別の時間、TeaCache ON/OFFなどを保存
- **成功したv8の推論構成（FP8-T5／INT8-DiT／SDPA互換パッチ／8ステップ）を維持**。G4高速・安定版・省VRAMを切替可能
- 完成MP4は再生成不要。⑨だけ再実行してZIPを再取得可能

**推奨：ColabでG4を選び、①から順に実行。** ②の`TEST_FIRST_SECONDS = 5`で短時間テスト、`0`で音声全体を使用。長尺は数時間以上になることがあり、Colabの連続稼働制限に注意してください。

**生成時間**は推論開始から生成動画の保存までを基準とします。初回セットアップ・モデルダウンロード・ファイル転送の時間は含めません。実際のGPU上での成功は未検証です。


## ① 画像と音声をアップロード（最初に実行）

アップロードが完了すると音声長を表示します。モデル取得を待つ必要はありません。

In [ ]:
# @title ① まず画像と音声をアップロード（モデルのダウンロードより先に実行）
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import subprocess, json

print('まず、画像1ファイル + 音声1ファイルを選択してください。')
print('対応画像: PNG/JPG/WEBP/BMP / 対応音声: WAV/MP3/M4A/AAC/FLAC/OGG/OPUS/WMA/MP4')
uploaded = files.upload()
image_ext = {'.png','.jpg','.jpeg','.webp','.bmp'}
audio_ext = {'.wav','.mp3','.m4a','.aac','.flac','.ogg','.opus','.wma','.mp4'}
imgs = [name for name in uploaded if Path(name).suffix.lower() in image_ext]
auds = [name for name in uploaded if Path(name).suffix.lower() in audio_ext]
assert len(imgs) == 1 and len(auds) == 1, f'画像1つ、音声1つを選んでください（画像{len(imgs)}・音声{len(auds)}）。'

UPLOAD_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
UPLOAD_DIR = Path('/content/infinitetalk_inputs') / UPLOAD_ID
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_IMAGE_SOURCE = UPLOAD_DIR / ('original_image' + Path(imgs[0]).suffix.lower())
UPLOAD_AUDIO_SOURCE = UPLOAD_DIR / ('original_audio' + Path(auds[0]).suffix.lower())
UPLOAD_IMAGE_SOURCE.write_bytes(uploaded[imgs[0]])
UPLOAD_AUDIO_SOURCE.write_bytes(uploaded[auds[0]])
UPLOAD_IMAGE_NAME = Path(imgs[0]).name
UPLOAD_AUDIO_NAME = Path(auds[0]).name
assert UPLOAD_IMAGE_SOURCE.stat().st_size > 0 and UPLOAD_AUDIO_SOURCE.stat().st_size > 0, 'ファイルが空です。'

# ffprobeでダウンロードを待たずに音声長を表示。PCMベースの厳密な秒数は③で算出。
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                        '-of','json',str(UPLOAD_AUDIO_SOURCE)],capture_output=True,text=True)
assert probe.returncode == 0, '音声を読み取れません。別の形式で試してください: ' + probe.stderr[-1000:]
UPLOAD_AUDIO_DURATION = float(json.loads(probe.stdout)['format']['duration'])
assert UPLOAD_AUDIO_DURATION >= 0.5, '音声が短すぎます（最低0.5秒）。'

print(f'✅ 画像: {UPLOAD_IMAGE_NAME} ({UPLOAD_IMAGE_SOURCE.stat().st_size/1024**2:.1f} MiB)')
print(f'✅ 音声: {UPLOAD_AUDIO_NAME} / 約{UPLOAD_AUDIO_DURATION:.2f}秒')
print('✅ アップロード完了。次は②設定・③音声準備。その後にモデルをダウンロードします。')
print('保管場所:', UPLOAD_DIR)


## ② 生成設定（音声は全長自動または5秒テスト）

`TEST_FIRST_SECONDS=0`なら音声全体。`5`なら最初の5秒でテスト。ZIP一括ダウンロードは初期設定でONです。

In [ ]:
# @title ② 生成オプション（音声の長さは自動検出）
PERFORMANCE_MODE = "G4高速"  # @param ["G4高速", "安定版", "省VRAM"]
TEST_FIRST_SECONDS = 0.0  # @param {type:"number"}
ENABLE_TEACACHE = False  # @param {type:"boolean"}
TEACACHE_THRESHOLD = 0.15  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
PROMPT = "A single anime-style young adult woman speaks naturally to the camera, accurate lip synchronization with her speech, expressive facial movements, subtle natural head movements, gentle upper-body movement and occasional small hand gestures. Keep the character identity, hair, outfit, camera angle and background consistent. No scene cuts, no extra people."  # @param {type:"string"}
AUTO_DOWNLOAD = True  # @param {type:"boolean"}
DOWNLOAD_MP4_SEPARATELY = False  # @param {type:"boolean"}

assert PERFORMANCE_MODE in ("G4高速", "安定版", "省VRAM")
assert 0 <= float(TEST_FIRST_SECONDS) <= 3600, "テスト制限は0〜3600秒。0なら音声の全長を使用します。"
assert 0 < float(TEACACHE_THRESHOLD) <= 1.0
assert isinstance(SEED, int)
print(f"✅ モード: {PERFORMANCE_MODE} / 音声長: 自動 / テスト上限: {TEST_FIRST_SECONDS}秒（0は無制限）")
print(f"✅ 480p・8ステップ / TeaCache: {'ON（試験的）' if ENABLE_TEACACHE else 'OFF'}")

print('✅ 完了後はMP4＋生成時間レポートのZIPをまとめてダウンロードします。')


## ③ 入力データを準備（画像・音声を変換し、生成ジョブを作成）

①でアップロード済みのファイルを使用。再アップロードは必要ありません。音声を16kHz WAVに変換し、生成長を自動計算します。

In [ ]:
from PIL import Image, ImageOps
from pathlib import Path
import subprocess, json, datetime, uuid, shutil, math, wave

assert 'TEST_FIRST_SECONDS' in globals(), '先に②生成設定を実行してください。'
assert 'UPLOAD_IMAGE_SOURCE' in globals() and Path(UPLOAD_IMAGE_SOURCE).is_file(), '先に①ファイルアップロードを実行してください。'
assert 'UPLOAD_AUDIO_SOURCE' in globals() and Path(UPLOAD_AUDIO_SOURCE).is_file(), '先に①ファイルアップロードを実行してください。'
job_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
JOB = Path('/content/infinitetalk_outputs') / job_id
JOB.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = JOB / 'reference.png'
AUDIO_PATH = JOB / 'speech_16k_mono.wav'

with Image.open(UPLOAD_IMAGE_SOURCE) as im:
    im = ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    im.save(IMAGE_PATH, optimize=True)

# 長さはffprobeメタデータではなく、16kHz PCMへ変換した後のサンプル数で測定。
# 0: 全長。テスト上限がある場合だけ意図的に先頭 N 秒を使用する。
ff = ['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error',
      '-i', str(UPLOAD_AUDIO_SOURCE), '-vn', '-ac','1','-ar','16000','-c:a','pcm_s16le']
if TEST_FIRST_SECONDS > 0:
    ff += ['-t', str(float(TEST_FIRST_SECONDS))]
ff += [str(AUDIO_PATH)]
subprocess.run(ff, check=True)

with wave.open(str(AUDIO_PATH), 'rb') as wf:
    assert wf.getframerate() == 16000 and wf.getnchannels() == 1
    TARGET_SEC = wf.getnframes() / wf.getframerate()
assert TARGET_SEC >= 0.5, f'有効な音声が短すぎます（{TARGET_SEC:.2f}秒）。0.5秒以上を用意してください。'

# モデルは81フレームより長い音声埋め込みを要求する。
# 4秒未満のみ内部入力を4.0秒までpadding。MP4仕上げは元のTARGET_SECで切り戻す。
MIN_MODEL_SEC = 4.0
if TARGET_SEC < MIN_MODEL_SEC:
    padded = JOB / 'speech_16k_model.wav'
    subprocess.run(['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error',
                    '-i', str(AUDIO_PATH), '-af', f'apad=whole_dur={MIN_MODEL_SEC}',
                    '-t', str(MIN_MODEL_SEC), '-ac','1','-ar','16000',
                    '-c:a','pcm_s16le', str(padded)], check=True)
    MODEL_AUDIO_PATH = padded
    print('ℹ️ 短い音声のため、モデル内部だけ4秒に無音延長。完成MP4は元の長さに戻します。')
else:
    MODEL_AUDIO_PATH = AUDIO_PATH
MODEL_SEC = max(TARGET_SEC, MIN_MODEL_SEC)
FRAME_COUNT = math.ceil(MODEL_SEC * 25)
# InfiniteTalkは最初81フレーム、後続は72フレームずつ増える。
CHUNK_COUNT = max(1, 1 + math.ceil(max(0, FRAME_COUNT - 81) / 72))

JSON_PATH = JOB / 'input.json'
JSON_PATH.write_text(json.dumps({
    'prompt': PROMPT, 'cond_video': str(IMAGE_PATH),
    'cond_audio': {'person1': str(MODEL_AUDIO_PATH)},
}, ensure_ascii=False, indent=2), encoding='utf-8')

META_PATH = JOB / 'job_meta.json'
META_PATH.write_text(json.dumps({
    'target_seconds': TARGET_SEC,
    'model_seconds': MODEL_SEC,
    'frame_count': FRAME_COUNT,
    'chunk_count_est': CHUNK_COUNT,
    'performance_mode': PERFORMANCE_MODE,
    'enable_teacache': ENABLE_TEACACHE,
    'test_limit_seconds': TEST_FIRST_SECONDS,
    'input_filename_image': UPLOAD_IMAGE_NAME,
    'source_upload_id': UPLOAD_ID,
    'input_filename_audio': UPLOAD_AUDIO_NAME,
}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'✅ 音声の長さ: {TARGET_SEC:.2f}秒 → 動画の目標長さも {TARGET_SEC:.2f}秒')
print(f'✅ 内部生成: {FRAME_COUNT}フレーム（約{CHUNK_COUNT}区間）')
print(f'✅ JOB: {JOB}')
print('✅ モデル取得前の入力準備が完了しました。')
if TARGET_SEC > 90:
    print('⚠ 長尺です。Colabランタイム切断・RAM不足や長時間の生成に注意。短時間テストを推奨します。')


## ④ GPU・空き容量を確認

In [ ]:
import subprocess, shutil, re, os, sys

def cmd_out(cmd):
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()

try:
    gpus = cmd_out(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'])
except Exception as e:
    raise RuntimeError('GPUが有効ではありません。Colabの「ランタイムのタイプを変更」でG4またはA100を選んでください。') from e
print('GPU:', gpus)
mem_mib = int(gpus.splitlines()[0].rsplit(',', 1)[-1].strip())
mem_gib = mem_mib / 1024
free_gib = shutil.disk_usage('/content').free / (1024**3)
print(f'VRAM: {mem_gib:.1f} GiB / ディスク空き: {free_gib:.1f} GiB')
print('Python:', sys.version.split()[0])
if mem_gib < 38:
    raise RuntimeError('この14Bモデルの40秒生成にはVRAMが不足しやすいです。G4またはA100を選んでください。')
if free_gib < 42:
    raise RuntimeError('モデル取得に必要なディスク空きが足りません。45〜55GiB程度の空きを確保してください。')
if mem_gib < 55:
    print('⚠ 40GiB前後のGPU：モデルをCPUへ退避する低VRAMモードを自動使用します。速度が低下します。')
print('✅ 事前チェック完了')

## ⑤ Python 3.10環境の準備（初回のみ）

以前の実行でモデルやPython環境が残っていれば再利用します。

In [ ]:
import os, sys, subprocess, pathlib, shutil, textwrap
ROOT = pathlib.Path('/content/InfiniteTalk')
VENV = pathlib.Path('/content/infinitetalk_py310')
VENV_PY = VENV / 'bin' / 'python'

# ログを省略しない。パッケージ名・依存エラーが見えるようにする。
def run(command, *, cwd=None):
    command = list(map(str, command))
    print('▶', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

# 初期化: 以前のColab Python 3.13向けpip失敗は無視して独立環境を使う。
os.environ['PIP_NO_CACHE_DIR'] = '1'
if not (ROOT / 'generate_infinitetalk.py').is_file():
    run(['git','clone','--depth','1','https://github.com/MeiGen-AI/InfiniteTalk.git',ROOT])
else:
    print('✅ 既存のInfiniteTalkリポジトリを再利用')

# uv はColabカーネルの Python 3.13 にインストールしても、モデル環境を汚さない。
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'uv'])
UV = shutil.which('uv')
assert UV, 'uvのインストールを確認できません。'

if not VENV_PY.exists():
    run([UV, 'python', 'install', '3.10'])
    run([UV, 'venv', '--python', '3.10', str(VENV)])
print('専用Python:', subprocess.check_output([str(VENV_PY), '-V'], text=True).strip())

# PyTorch/xFormersのABI/CUDA互換性を揃える（G4 Blackwell向けCUDA 12.8）。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '--index-url', 'https://download.pytorch.org/whl/cu128',
     'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1'])

# ColabではGradio UIやDashscope APIを使わないため取り除き、依存関係の将来変更を回避。
# xfuserは単一GPUでも一部の公式コードからimportされるので2025年互換版を維持。
REQ = ROOT / 'requirements.colab.txt'
REQ.write_text('\n'.join(line for line in (ROOT / 'requirements.txt').read_text().splitlines()
                       if not line.strip().startswith(('gradio', 'dashscope', 'xfuser'))) + '\n')
# torchの版を固定し、インストール時のGPUパッケージ入れ替えを防ぐ。
run([UV, 'pip', 'install', '--python', str(VENV_PY),
     '-r', str(REQ), 'xfuser==0.4.4',
     'torch==2.7.1', 'xformers==0.0.31.post1',
     'numpy==1.26.4', 'opencv-python==4.10.0.84',
     'transformers==4.49.0', 'diffusers==0.33.1',
     'huggingface_hub==0.34.4', 'librosa==0.10.2.post1',
     'misaki[en]', 'soundfile', 'hf-xet', 'ninja'])

# FP8 T5をGPUに転送する際、optimum-quantoがC++/CUDA拡張をJITビルドする。
# uvが作った仮想環境にはpipが入っていない場合があるため、必ずuv pipでインストールする。
# venv/binを PATH の先頭に追加し、torch.utils.cpp_extension がninjaを見つけられるようにする。
os.environ['PATH'] = str(VENV / 'bin') + os.pathsep + os.environ.get('PATH', '')
run(['ninja', '--version'])
print('✅ Ninja を発見。CUDA拡張のビルドに利用できます。')
# nvccがない場合は、次のGPU生成セルでエラーになる可能性がある。ここで状況を可視化。
if shutil.which('nvcc'):
    result = subprocess.run(['nvcc', '--version'], text=True, capture_output=True)
    print('CUDA compiler:', (result.stdout.strip().splitlines() or ['unknown'])[-1])
else:
    print('⚠️ nvccが PATH に見つかりません。次にCUDA toolkit関連のエラーが出る可能性があります。')

if shutil.which('ffmpeg') is None:
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','ffmpeg'])

# Pythonのバージョン変更に対応する公式ソースの最小限の互換性パッチ。
# パッチは繰り返し実行しても変化しない（冪等）。
mt = ROOT / 'wan' / 'multitalk.py'
s = mt.read_text()
if 'from inspect import ArgSpec' in s:
    s = s.replace('from inspect import ArgSpec', 'from inspect import FullArgSpec as ArgSpec')
    mt.write_text(s)
    print('✅ inspect.ArgSpec互換パッチを適用')
wv = ROOT / 'src' / 'audio_analysis' / 'wav2vec2.py'
s = wv.read_text()
needle = '        self.config.output_attentions = True'
if needle in s and '_attn_implementation = "eager"' not in s:
    s = s.replace(needle, '        self.config._attn_implementation = "eager"\n' + needle)
    wv.write_text(s)
    print('✅ Wav2Vec2 eager attention互換パッチを適用')

# エラーを早めに検出。OSカーネルではなく専用Pythonで実行。
run([str(VENV_PY), '-c',
     'import torch, torchvision, xformers, transformers, diffusers, numpy; '
     'import wan.multitalk; '
     'print("環境OK: torch", torch.__version__, "cuda", torch.version.cuda, "numpy", numpy.__version__)'],
    cwd=ROOT)
print('✅ 環境構築完了。Colabランタイムの再起動は不要です。')


## ⑤-B SDPA互換パッチ（必要）

以前のFlashAttentionエラーを回避するため、既存の成功構成を維持します。

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
attn_file = ROOT / 'wan' / 'modules' / 'attention.py'
assert attn_file.is_file(), '②でInfiniteTalkをセットアップしてください。'
assert VENV_PY.is_file(), '②でPython 3.10環境を構築してください。'

s = attn_file.read_text(encoding='utf-8')
marker = '# COLAB_SDPA_FALLBACK_V5'
needle = '    half_dtypes = (torch.float16, torch.bfloat16)'

fallback = '''    # COLAB_SDPA_FALLBACK_V5
    # FlashAttention 2 is unavailable in some Colab runtimes (including G4).
    # Use PyTorch SDPA while preserving the [B, seq, heads, dim] API and dtype.
    if not FLASH_ATTN_2_AVAILABLE and (version == 2 or not FLASH_ATTN_3_AVAILABLE):
        b, lq, n, d = q.shape
        lk = k.shape[1]
        out_dtype = q.dtype
        work_dtype = v.dtype if v.dtype in (torch.float16, torch.bfloat16) else dtype
        qq = q.transpose(1, 2).to(dtype=work_dtype)
        kk = k.transpose(1, 2).to(dtype=work_dtype)
        vv = v.transpose(1, 2).to(dtype=work_dtype)
        if q_scale is not None:
            qq = qq * q_scale

        mask = None
        if k_lens is not None:
            kl = torch.as_tensor(k_lens, device=q.device).reshape(b)
            valid_keys = torch.arange(lk, device=q.device)[None, None, None, :] < kl[:, None, None, None]
            mask = valid_keys
        if window_size is not None and tuple(window_size) != (-1, -1):
            left, right = window_size
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            local = torch.ones((1, 1, lq, lk), device=q.device, dtype=torch.bool)
            if left >= 0:
                local = local & (ki >= qi - left)
            if right >= 0:
                local = local & (ki <= qi + right)
            mask = local if mask is None else mask & local
        # Build an explicit causal mask when necessary to combine with key masking.
        if causal:
            qi = torch.arange(lq, device=q.device)[None, None, :, None]
            ki = torch.arange(lk, device=q.device)[None, None, None, :]
            # FlashAttention varlen causal alignment is bottom-right for Lq != Lk.
            causal_mask = ki <= qi + (lk - lq)
            mask = causal_mask if mask is None else mask & causal_mask

        result = torch.nn.functional.scaled_dot_product_attention(
            qq, kk, vv, attn_mask=mask, dropout_p=dropout_p,
            is_causal=False, scale=softmax_scale)
        if q_lens is not None:
            ql = torch.as_tensor(q_lens, device=q.device).reshape(b)
            valid_q = torch.arange(lq, device=q.device)[None, None, :, None] < ql[:, None, None, None]
            result = result.masked_fill(~valid_q, 0)
        return result.transpose(1, 2).contiguous().to(out_dtype)

'''

if marker in s:
    print('✅ SDPAパッチは既に適用済み（再実行しても問題ありません）')
else:
    assert needle in s, 'attention.pyのコード構造が変わっています。パッチ適用を中止しました。'
    s = s.replace(needle, fallback + needle, 1)
    # Syntax validation before modifying checked-out source
    compile(s, str(attn_file), 'exec')
    attn_file.with_suffix('.py.before_colab_sdpa_v5').write_text(attn_file.read_text(encoding='utf-8'), encoding='utf-8')
    attn_file.write_text(s, encoding='utf-8')
    print('✅ FlashAttention不在時のPyTorch SDPAフォールバックを追加')

# New subprocess: no runtime restart. Two tests including key padding and causal behavior.
probe = r'''
import torch
from wan.modules.attention import flash_attention
assert torch.cuda.is_available(), 'GPUが有効ではありません'
dev = 'cuda'
for lq, lk, causal in [(8, 8, False), (5, 9, True)]:
    q = torch.randn(2, lq, 4, 32, device=dev, dtype=torch.float16)
    k = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    v = torch.randn(2, lk, 4, 32, device=dev, dtype=torch.float16)
    k_lens = torch.tensor([lk, lk - 1], device=dev, dtype=torch.int32)
    out = flash_attention(q, k, v, k_lens=k_lens, causal=causal, version=2)
    assert out.shape == q.shape and out.dtype == q.dtype
    assert torch.isfinite(out).all()
print('✅ Attention動作テスト成功 / GPU:', torch.cuda.get_device_name(0))
'''
subprocess.run([str(VENV_PY), '-c', probe], cwd=str(ROOT), check=True)
print('✅ このあと⑤で画像と音声を用意し、⑥→⑦を実行してください。')


## ⑥ モデルをダウンロード（初回のみ）

**画像と音声はすでにアップロード済みです。** 長いダウンロードをここから開始します。モデルが残っていれば再利用します。

In [ ]:
from pathlib import Path
import os, subprocess, shutil, textwrap
ROOT = Path('/content/InfiniteTalk')
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '②の環境構築から実行してください。'
W = ROOT / 'weights'
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
QUANT_NAME = 'infinitetalk_single_int8_lora'
QUANT = IT / 'quant_models' / (QUANT_NAME + '.safetensors')

free = shutil.disk_usage('/content').free / 1024**3
if not QUANT.exists() and free < 32:
    raise RuntimeError(f'ディスク空き {free:.1f} GiB。容量不足の可能性。')

# HFライブラリも仮想環境側。ノートブックカーネルへインポートしない。
download_script = r'''
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download
W = Path('/content/InfiniteTalk/weights')
BASE = W / 'Wan2.1-I2V-14B-480P'
IT = W / 'InfiniteTalk'
WV = W / 'chinese-wav2vec2-base'
print('1/3: Wan2.1補助モデル', flush=True)
snapshot_download('Wan-AI/Wan2.1-I2V-14B-480P',local_dir=str(BASE),
  allow_patterns=['config.json','Wan2.1_VAE.pth',
   'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
   'google/**','xlm-roberta-large/**'],max_workers=3)
print('2/3: InfiniteTalk INT8+LoRA / FP8 T5', flush=True)
snapshot_download('MeiGen-AI/InfiniteTalk',local_dir=str(IT),
  allow_patterns=['quant_models/infinitetalk_single_int8_lora.safetensors',
   'quant_models/infinitetalk_single_int8_lora.json',
   'quant_models/t5_fp8.safetensors','quant_models/t5_map_fp8.json'],max_workers=3)
print('3/3: 音声エンコーダー', flush=True)
snapshot_download('TencentGameMate/chinese-wav2vec2-base',local_dir=str(WV),
  allow_patterns=['*.json','*.txt','*.model'], max_workers=3)
hf_hub_download('TencentGameMate/chinese-wav2vec2-base', filename='model.safetensors',
  revision='refs/pr/1',local_dir=str(WV))
print('✅ ダウンロード完了', flush=True)
'''
subprocess.run([str(VENV_PY), '-c', download_script], cwd=str(ROOT), check=True)

needed = [BASE/'config.json', BASE/'Wan2.1_VAE.pth',
 BASE/'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
 QUANT, IT/'quant_models'/'infinitetalk_single_int8_lora.json',
 IT/'quant_models'/'t5_fp8.safetensors', IT/'quant_models'/'t5_map_fp8.json',
 WV/'model.safetensors']
missing = [str(p) for p in needed if not p.is_file()]
assert not missing, '必須ファイル不足（③を再実行）:\n' + '\n'.join(missing)
for p in needed:
    print(f'✅ {p.relative_to(ROOT)} ({p.stat().st_size / 1024**2:,.1f} MiB)')


## ⑦ モデルファイルのセルフチェック（任意）

In [ ]:
from pathlib import Path
import subprocess, shutil
ROOT = Path('/content/InfiniteTalk')
BASE = ROOT / 'weights' / 'Wan2.1-I2V-14B-480P'
IT = ROOT / 'weights' / 'InfiniteTalk'
WV = ROOT / 'weights' / 'chinese-wav2vec2-base'
required = {
    'Wan config': BASE / 'config.json',
    'Wan VAE': BASE / 'Wan2.1_VAE.pth',
    'Wan CLIP': BASE / 'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
    'INT8+LoRA DiT': IT / 'quant_models' / 'infinitetalk_single_int8_lora.safetensors',
    'INT8+LoRA map': IT / 'quant_models' / 'infinitetalk_single_int8_lora.json',
    'FP8 T5': IT / 'quant_models' / 't5_fp8.safetensors',
    'FP8 T5 map': IT / 'quant_models' / 't5_map_fp8.json',
    'Audio encoder': WV / 'model.safetensors',
}
print('== GPU ==')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],check=False)
print('== ディスク ==')
print(f"{shutil.disk_usage('/content').free / 1024**3:.1f} GiB 空き")
print('== チェックポイント ==')
failed=[]
for key,path in required.items():
    ok = path.is_file() and path.stat().st_size > 0
    label = f'{path.stat().st_size / 1024**3:.2f} GiB' if ok else '見つからない / サイズ0'
    print(('✅' if ok else '❌'),key,':',label)
    if not ok: failed.append(str(path))
if failed:
    print('\nモデルファイルが不足しています。③のセルを再実行してください。')
else:
    print('\n✅ 基本ファイルは揃っています。')

## ⑧ InfiniteTalkを生成（進捗・時間計測）

モデル読み込み＋推論の時間、区間別の時間をログとJSON/CSVに記録します。

In [ ]:
from pathlib import Path
import subprocess, os, json, shutil, math, time, datetime, uuid, csv, re
# 既存のColabランタイムを再利用した場合もモデルパスを再構成する。
ROOT = Path('/content/InfiniteTalk')
BASE = ROOT / 'weights' / 'Wan2.1-I2V-14B-480P'
WV = ROOT / 'weights' / 'chinese-wav2vec2-base'
QUANT = ROOT / 'weights' / 'InfiniteTalk' / 'quant_models' / 'infinitetalk_single_int8_lora.safetensors'
assert Path(JSON_PATH).exists(), '先に③音声準備を実行してください。'
assert QUANT.exists(), '先に⑥モデル取得を実行してください。'
assert FRAME_COUNT >= 81 and TARGET_SEC >= 0.5
RAW_PREFIX = JOB / 'infinitetalk_raw'
LOG_FILE = JOB / 'generate.log'
VENV_PY = Path('/content/infinitetalk_py310/bin/python')
assert VENV_PY.exists(), '⑤環境構築を実行してください。'

# 実際のGPU空き容量をチェック。G4約96GBで余裕がある場合だけ高速モード。
try:
    gpu_csv = subprocess.check_output(
        ['nvidia-smi','--query-gpu=memory.total,memory.free','--format=csv,noheader,nounits'],
        text=True).splitlines()[0]
    total_mib, free_mib = [int(v.strip()) for v in gpu_csv.split(',')]
except Exception as e:
    raise RuntimeError('GPUが見つかりません。④でGPUを確認してください。') from e
vram_total_gib, vram_free_gib = total_mib / 1024, free_mib / 1024

requested_mode = PERFORMANCE_MODE
actual_mode = requested_mode
if requested_mode == 'G4高速' and (vram_total_gib < 70 or vram_free_gib < 65):
    actual_mode = '安定版'
    print(f'⚠ 空きVRAM {vram_free_gib:.1f} GiB：G4高速は危険なため安定版に切替。',flush=True)

offload = 'False' if actual_mode == 'G4高速' else 'True'
args = [str(VENV_PY), 'generate_infinitetalk.py',
        '--ckpt_dir', str(BASE),
        '--wav2vec_dir', str(WV),
        '--quant_dir', str(QUANT),
        '--quant', 'fp8',  # v6で実績のある量子化復元設定（INT8 DiTのマップはファイル側）
        '--input_json', str(JSON_PATH),
        '--size', 'infinitetalk-480',
        '--sample_steps', '8',
        '--sample_text_guide_scale', '1.0',
        '--sample_audio_guide_scale', '2.0',
        '--sample_shift', '2',
        '--mode', 'streaming',
        '--frame_num', '81',
        '--max_frame_num', str(int(FRAME_COUNT)),
        '--motion_frame', '9',
        '--base_seed', str(SEED),
        '--offload_model', offload,
        '--save_file', str(RAW_PREFIX),
        '--audio_save_dir', str(JOB/'audio_embeddings')]
if actual_mode == '省VRAM' or vram_total_gib < 55:
    args += ['--num_persistent_param_in_dit', '0', '--t5_cpu']
if ENABLE_TEACACHE:
    args += ['--use_teacache', '--teacache_thresh', str(float(TEACACHE_THRESHOLD))]

print(f'✅ 動画目標: {TARGET_SEC:.2f}秒 / {FRAME_COUNT}フレーム / 約{CHUNK_COUNT}区間',flush=True)
print(f'✅ モード: {actual_mode} / offload={offload} / TeaCache={ENABLE_TEACACHE}',flush=True)
print(f'✅ GPU空き: {vram_free_gib:.1f}/{vram_total_gib:.1f} GiB',flush=True)

# v3環境を同じColabセッションで使い回しても修復できるようにする。
# torch の拡張ロードでは Ninja が必要。uv venvにはpip自体がない場合もある。
import sys
VENV_BIN = VENV_PY.parent
if not (VENV_BIN / 'ninja').exists():
    uv_exe = shutil.which('uv')
    assert uv_exe, 'uvが見つかりません。先に②の環境構築を実行してください。'
    print('Ninjaがないため、専用環境に自動インストールします…', flush=True)
    subprocess.run([uv_exe, 'pip', 'install', '--python', str(VENV_PY), 'ninja'], check=True)

runtime_env = os.environ.copy()
runtime_env['PATH'] = str(VENV_BIN) + os.pathsep + runtime_env.get('PATH', '')
# ColabでGPU拡張の同時ビルド数が増えすぎないようにする。
runtime_env['MAX_JOBS'] = '2'
check_ninja = subprocess.run([str(VENV_BIN / 'ninja'), '--version'],
                             capture_output=True, text=True, env=runtime_env)
assert check_ninja.returncode == 0, f'Ninja が実行できません: {check_ninja.stderr}'
print('✅ Ninja:', check_ninja.stdout.strip(), flush=True)

runtime_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
runtime_env['TOKENIZERS_PARALLELISM'] = 'false'
# 同一ジョブの⑥を再実行しても古いログと計測値を残す。
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
if LOG_FILE.is_file():
    old_log = JOB / f'generate_previous_{RUN_ID}.log'
    shutil.copy2(LOG_FILE, old_log)
METRICS_FILE = JOB / f'run_metrics_{RUN_ID}.json'
METRICS_LATEST = JOB / 'metrics_latest.json'
METRICS_CSV = Path('/content/infinitetalk_outputs/generation_metrics.csv')

# 各ジョブのJSONを集約してCSVを再構築：同じ⑦を再実行しても重複しない。
def refresh_metrics_csv():
    columns = [
        'started_utc','run_id','job_id','state','gpu','mode','offload',
        'teacache','teacache_threshold','target_seconds','frame_count',
        'estimated_chunks','completed_chunks','sample_steps',
        'generation_seconds','chunk_seconds','seconds_per_video_second',
        'finalization_seconds','combined_processing_seconds','log_file','final_video'
    ]
    rows = []
    for p in sorted(METRICS_CSV.parent.glob('*/run_metrics_*.json')):
        try:
            entry = json.loads(p.read_text(encoding='utf-8'))
            rows.append({
                'started_utc': entry.get('started_utc',''),
                'run_id': entry.get('run_id',''),
                'job_id': entry.get('job_id',''),
                'state': entry.get('state',''),
                'gpu': entry.get('gpu',''),
                'mode': entry.get('mode',''),
                'offload': entry.get('offload',''),
                'teacache': entry.get('teacache',''),
                'teacache_threshold': entry.get('teacache_threshold',''),
                'target_seconds': entry.get('target_seconds',''),
                'frame_count': entry.get('frame_count',''),
                'estimated_chunks': entry.get('estimated_chunks',''),
                'completed_chunks': len(entry.get('chunk_seconds',[])),
                'sample_steps': entry.get('sample_steps',''),
                'generation_seconds': entry.get('generation_seconds',''),
                'chunk_seconds': '|'.join(str(x) for x in entry.get('chunk_seconds',[])),
                'seconds_per_video_second': entry.get('seconds_per_video_second',''),
                'finalization_seconds': entry.get('finalization_seconds',''),
                'combined_processing_seconds': entry.get('combined_processing_seconds',''),
                'log_file': entry.get('log_file',''),
                'final_video': entry.get('final_video',''),
            })
        except Exception as exc:
            print('⚠️ 読み込めない計測JSONはスキップ:', p, exc)
    temp = METRICS_CSV.with_suffix('.csv.tmp')
    with temp.open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(METRICS_CSV)


def clock_hms(sec):
    s = int(round(max(0.0, float(sec))))
    return f'{s//3600:02d}:{(s//60)%60:02d}:{s%60:02d}'

# 計測は⑧の生成プロセス開始から。⑥モデル取得や①アップロードは含まない。
started_at_utc = datetime.datetime.now(datetime.timezone.utc)
start_perf = time.perf_counter()
status = None
interrupt = False
chunk_times = []
chunk_active = False
chunk_started = None
last_progress = -1
proc = None
print('生成開始（モデル読み込みと推論に時間がかかります）。出力:', RAW_PREFIX, flush=True)
try:
    with open(LOG_FILE, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(args, cwd=str(ROOT), env=runtime_env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                # tqdmはCR(\r)を使って更新するため分割して認識。最後の8/8重複は無視。
                for part in re.split(r'[\r\n]+', line):
                    found = re.search(r'(?<!\d)([0-8])/8\s*\[', part)
                    if not found:
                        continue
                    step = int(found.group(1))
                    now = time.perf_counter()
                    if step < 8 and not chunk_active:
                        chunk_active = True
                        chunk_started = now
                    elif step == 8 and chunk_active:
                        chunk_times.append(round(now - chunk_started, 2))
                        chunk_active = False
                        done = len(chunk_times)
                        total = time.perf_counter() - start_perf
                        remaining_chunks = max(0, int(CHUNK_COUNT) - done)
                        eta = (sum(chunk_times)/done) * remaining_chunks if done else 0
                        print(f'\n⏱️ 区間{done}/{CHUNK_COUNT} 完了：'
                              f'{clock_hms(chunk_times[-1])} / '
                              f'生成経過 {clock_hms(total)} / '
                              f'残り目安 {clock_hms(eta)}', flush=True)
            status = proc.wait()
        except KeyboardInterrupt:
            interrupt = True
            proc.terminate()
            try:
                status = proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                proc.kill()
                status = proc.wait()
            raise
finally:
    finished_at_utc = datetime.datetime.now(datetime.timezone.utc)
    elapsed = round(time.perf_counter() - start_perf, 2)
    raw_output = Path(str(RAW_PREFIX) + '.mp4')
    success = status == 0 and raw_output.exists() and raw_output.stat().st_size > 10_000
    state = 'interrupted' if interrupt else ('success' if success else 'failed')
    try:
        gpu_name = subprocess.check_output(
            ['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
            text=True,timeout=10).strip().splitlines()[0]
    except Exception:
        gpu_name = 'unknown'
    data = {
        'run_id': RUN_ID,
        'job_id': JOB.name,
        'started_utc': started_at_utc.isoformat(),
        'finished_utc': finished_at_utc.isoformat(),
        'state': state,
        'exit_code': status,
        'gpu': gpu_name,
        'mode': actual_mode,
        'offload': offload,
        'teacache': bool(ENABLE_TEACACHE),
        'teacache_threshold': float(TEACACHE_THRESHOLD) if ENABLE_TEACACHE else None,
        'sample_steps': 8,
        'target_seconds': round(float(TARGET_SEC),3),
        'frame_count': int(FRAME_COUNT),
        'estimated_chunks': int(CHUNK_COUNT),
        'chunk_seconds': chunk_times,
        'generation_seconds': elapsed,
        'seconds_per_video_second': round(elapsed / TARGET_SEC,3),
        'log_file': str(LOG_FILE),
        'raw_video': str(raw_output) if success else None,
        'final_video': None,
        'finalization_seconds': None,
        'combined_processing_seconds': None,
    }
    try:
        METRICS_FILE.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        METRICS_LATEST.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        refresh_metrics_csv()
    except Exception as exc:
        print('⚠️ 計測ファイルの保存で問題が発生:',repr(exc))
    print('\n' + '='*16 + ' ⑧ 生成時間レポート ' + '='*16, flush=True)
    print('結果:',state,'/ 終了コード:',status,flush=True)
    print('モデル読み込み・生成の総時間:', clock_hms(elapsed),
          f'({elapsed/60:.2f}分)',flush=True)
    print('完了した区間:',f'{len(chunk_times)}/{CHUNK_COUNT}',flush=True)
    if chunk_times:
        print('区間別:', ', '.join(f'{i+1}={clock_hms(v)}' for i,v in enumerate(chunk_times)),flush=True)
    print('TeaCache:',ENABLE_TEACACHE,'/ offload:',offload,flush=True)
    print('計測CSV:',METRICS_CSV,flush=True)
    print('='*51,flush=True)

if status != 0:
    # 終了コードだけでは何が失敗したか分からない。常に元のエラーを表示する。
    import re
    lines = LOG_FILE.read_text(encoding='utf-8', errors='replace').splitlines()
    # ANSIカラーコードなどを除去し、エラー解析を読みやすくする。
    clean = [re.sub(r'\x1b\[[0-9;]*m', '', line) for line in lines]
    patterns = [r'Traceback', r'Error:', r'Exception:', r'CUDA out of memory',
                r'No such file', r'not found', r'unsupported', r'AssertionError',
                r'safetensors', r'Killed', r'undefined symbol', r'cannot import',
                r'Failed', r'RuntimeError', r'KeyError']
    interesting = [i for i, line in enumerate(clean)
                   if any(re.search(pat, line, re.I) for pat in patterns)]
    print('\n'+'='*24+' InfiniteTalk 失敗診断 '+'='*24)
    print('終了コード:', status)
    print('ログ:', LOG_FILE)
    if interesting:
        print('\n▼ 検出したエラー箇所（前後の文脈を含む）')
        a, b = max(0,interesting[-1]-24), min(len(clean),interesting[-1]+24)
        print('\n'.join(f'{i+1}: {clean[i]}' for i in range(a,b)))
    print('\n▼ ログ末尾（最大100行）')
    print('\n'.join(clean[-100:]))
    try:
        gpu_report = subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.free,memory.total',
                                     '--format=csv,noheader'], capture_output=True,text=True)
        print('\nGPU 状態:',gpu_report.stdout.strip() or gpu_report.stderr.strip())
    except Exception as ex:
        print('GPU診断不可:',ex)
    text_lower = '\n'.join(clean[-200:]).lower()
    if any(x in text_lower for x in ('out of memory','cuda error: out of memory', 'cuda oom')):
        print('\n推奨：GPUメモリ不足です。②を安定版か省VRAMに変え、⑤から再試行。')
    elif 'ninja is required' in text_lower:
        print('\n推奨：Ninjaが見つかりません。⑤を再実行するかv4ノートブックで再実行してください。')
    elif any(x in text_lower for x in ('nvcc', 'cuda_home', 'cuda toolkit')):
        print('\n推奨：CUDA拡張のビルドでエラー。ログ内のCUDA compiler / nvcc の記述を確認してください。')
    elif any(x in text_lower for x in ('no such file', 'filenotfounderror','entry not found')):
        print('\n推奨：モデルやファイルの欠損が考えられます。③のダウンロードを再実行してください。')
    elif any(x in text_lower for x in ('modulenotfounderror','importerror','undefined symbol')):
        print('\n推奨：ライブラリの互換性問題が考えられます。ログのエラー行を共有してください。')
    elif any(x in text_lower for x in ('no space left on device', 'enospc')):
        print('\n推奨：Colabのディスク空き容量を確認してください。')
    else:
        print('\n推奨：ログ末尾を確認。必要なら②を安定版に切り替えて再試行してください。')
    raise RuntimeError(f'InfiniteTalk生成失敗（終了コード {status}）。上に実際のエラーとログ末尾を表示しました。')
RAW_VIDEO = Path(str(RAW_PREFIX) + '.mp4')
assert RAW_VIDEO.exists() and RAW_VIDEO.stat().st_size > 10_000, f'出力動画がありません: {RAW_VIDEO}'
print('✅ 推論完了:', RAW_VIDEO)

## ⑨ MP4仕上げ・生成時間レポート・ZIP一括ダウンロード

**MP4と生成にかかった時間を1つのZIPで保存します。** 生成済み動画があれば再生成せず、このセルのみで取得できます。

In [ ]:
# @title ⑨ 生成MP4の仕上げ＋生成時間レポート＋ZIP一括ダウンロード（再生成なし）
from pathlib import Path
from IPython.display import display, HTML, FileLink
from google.colab import files
from datetime import datetime, timezone
import subprocess, json, base64, time, zipfile

finish_started = time.perf_counter()
output_root = Path('/content/infinitetalk_outputs')
if 'JOB' in globals() and Path(JOB).is_dir() and (
    (Path(JOB) / 'infinitetalk_raw.mp4').is_file() or
    list(Path(JOB).glob('InfiniteTalk_*s_*.mp4'))):
    current_job = Path(JOB)
else:
    available = [p for p in output_root.glob('*') if p.is_dir() and
                 (list(p.glob('InfiniteTalk_*s_*.mp4')) or (p / 'infinitetalk_raw.mp4').is_file())]
    assert available, '生成された動画がありません。⑧が正常終了しているか確認してください。'
    current_job = max(available, key=lambda p: p.stat().st_mtime)
    print('ℹ️ 既存の生成ジョブを選択:', current_job.name)

raw_video = current_job / 'infinitetalk_raw.mp4'
audio_path = current_job / 'speech_16k_mono.wav'

def video_info(path):
    if not path.is_file() or path.stat().st_size < 1024:
        return None
    try:
        p = subprocess.run(['ffprobe', '-v', 'error', '-show_entries','format=duration,size',
             '-show_entries','stream=codec_type,codec_name','-of','json',str(path)],
             capture_output=True,text=True,timeout=45,check=True)
        d = json.loads(p.stdout)
        secs = float(d.get('format',{}).get('duration',0))
        if secs <= 0 or not any(s.get('codec_type')=='video' for s in d.get('streams', [])):
            return None
        return {'duration':secs,'size':path.stat().st_size}
    except Exception:
        return None

meta_path = current_job / 'job_meta.json'
if meta_path.is_file():
    job_meta = json.loads(meta_path.read_text(encoding='utf-8'))
    target_seconds = float(job_meta['target_seconds'])
elif audio_path.is_file():
    p = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                        '-of','json',str(audio_path)],capture_output=True,text=True,check=True)
    target_seconds = float(json.loads(p.stdout)['format']['duration'])
else:
    target_seconds = 0.0

token = f'{target_seconds:.2f}'.rstrip('0').rstrip('.') if target_seconds else 'saved'
target = current_job / f'InfiniteTalk_{token}s_{current_job.name}.mp4'
existing = video_info(target)
if existing and (target_seconds == 0 or abs(existing['duration'] - target_seconds) <= 0.5):
    print('✅ 保存済みの完成動画を再利用:',target.name)
else:
    others = [(p,video_info(p)) for p in current_job.glob('InfiniteTalk_*s_*.mp4') if p != target]
    others = [(p,info) for p,info in others if info]
    if others and (not video_info(raw_video) or not meta_path.is_file()):
        target, existing = max(others,key=lambda x:x[0].stat().st_mtime)
        print('✅ 既存MP4を再利用:',target.name)
    else:
        assert video_info(raw_video), '元動画がありません。⑧の生成が完了したか確認してください。'
        assert audio_path.is_file(), '仕上げに使う音声が見つかりません。'
        temp = current_job / f'.InfiniteTalk_{token}s_{current_job.name}.part.mp4'
        if temp.exists():temp.unlink()
        cmd = ['ffmpeg','-y','-nostdin','-hide_banner','-loglevel','error',
               '-i',str(raw_video),'-i',str(audio_path),
               '-filter_complex',f'[0:v]fps=25,tpad=stop_mode=clone:stop_duration=3,'
                                 f'trim=duration={target_seconds},setpts=PTS-STARTPTS[v]',
               '-map','[v]','-map','1:a:0', '-c:v','libx264','-preset','veryfast',
               '-crf','20','-pix_fmt','yuv420p','-c:a','aac','-b:a','192k',
               '-t',str(target_seconds),'-movflags','+faststart',str(temp)]
        r = subprocess.run(cmd,capture_output=True,text=True)
        if r.returncode != 0 or not video_info(temp):
            raise RuntimeError('FFmpegで仕上げ失敗。元動画は保存したままです: '+(r.stderr or '')[-1500:])
        temp.replace(target)
        print('✅ MP4仕上げ完了')

info = video_info(target)
assert info, '完成MP4を検証できません。'
print(f"✅ 完成動画: {target.name} / {info['duration']:.2f}秒 / {info['size']/1024**2:.1f} MiB")

# 計測は⑧の推論時間＋⑨の仕上げ（ダウンロードやプレビューの待機は含まない）。
finish_elapsed = round(time.perf_counter() - finish_started,2)
latest_meta = current_job / 'metrics_latest.json'
metrics = {}
if latest_meta.is_file():
    try:
        metrics = json.loads(latest_meta.read_text(encoding='utf-8'))
    except Exception as exc:
        print('⚠️ 計測データの読み込みに失敗:',exc)
if metrics:
    if metrics.get('finalization_seconds') is None:
        metrics['finalization_seconds'] = finish_elapsed
        metrics['combined_processing_seconds'] = round(
            float(metrics['generation_seconds'])+finish_elapsed,2)
    metrics['final_video']=str(target)
    for path in (latest_meta, current_job / f"run_metrics_{metrics['run_id']}.json"):
        try:path.write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
        except Exception as exc:print('⚠️ 計測JSONを書き込めません:',path,exc)
    if 'refresh_metrics_csv' in globals():
        try:refresh_metrics_csv()
        except Exception as exc:print('⚠️ 集計CSVの更新失敗:',exc)
else:
    print('⚠️ 過去の生成などで時間記録がありません。実測値は推測せず「記録なし」とします。')

def hms(v):
    if v is None:return '記録なし'
    s = max(0,int(round(float(v))))
    return f'{s//3600:02d}:{(s//60)%60:02d}:{s%60:02d}'

chunks = metrics.get('chunk_seconds',[])
lines = [
    'InfiniteTalk 生成時間レポート',
    '==============================',
    f'動画ファイル: {target.name}',
    f'動画の長さ: {info["duration"]:.2f} 秒',
    f'目標の長さ: {target_seconds:.2f} 秒',
    f'生成ジョブID: {current_job.name}',
    f'実行日時(UTC): {metrics.get("started_utc","記録なし")}',
    f'GPU: {metrics.get("gpu","記録なし")}',
    f'実行モード: {metrics.get("mode","記録なし")}',
    f'TeaCache: {metrics.get("teacache","記録なし")}',
    f'CPUオフロード: {metrics.get("offload","記録なし")}',
    '',
    f'⑧ モデル読み込み＋推論＋生成動画保存: {hms(metrics.get("generation_seconds"))}',
    f'⑨ MP4仕上げ: {hms(metrics.get("finalization_seconds"))}',
    f'⑧＋⑨ 合計: {hms(metrics.get("combined_processing_seconds"))}',
    '※ モデルの初回ダウンロード、音声アップロード、ブラウザダウンロード時間は含まれません。',
    '', '区間ごとの実測推論時間（目安）:',
]
lines.extend([f'  区間{i}: {hms(t)} ({float(t):.2f} 秒)' for i,t in enumerate(chunks,1)])
if not chunks:lines.append('  記録なし')
report_txt = current_job / 'generation_time_report.txt'
report_json = current_job / 'generation_time_report.json'
report_txt.write_text('\n'.join(lines)+'\n',encoding='utf-8-sig')
report_json.write_text(json.dumps({
    'video_filename': target.name, 'video_duration_seconds': info['duration'],
    'target_seconds':target_seconds, 'report_generated_at_utc':datetime.now(timezone.utc).isoformat(),
    'timing':metrics if metrics else None,
    'note':'Timing excludes initial setup/model downloads and browser transfers.'
},ensure_ascii=False,indent=2),encoding='utf-8')
print('\n'.join(lines[:19]))

# 一回のブラウザ操作でMP4と時間データをまとめて取得。
# MP4は既に圧縮済みなので ZIP_STORED でコピー速度を優先。
zip_path = current_job / f'InfiniteTalk_{current_job.name}_MP4_and_Timing.zip'
zip_tmp = zip_path.with_suffix('.zip.tmp')
try:
    with zipfile.ZipFile(zip_tmp,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for p in (target,report_txt,report_json,latest_meta,
                  Path('/content/infinitetalk_outputs/generation_metrics.csv')):
            if p.is_file():
                z.write(p,arcname=p.name)
    with zipfile.ZipFile(zip_tmp) as z:
        broken = z.testzip()
        assert broken is None, f'ZIP内の破損ファイル: {broken}'
    zip_tmp.replace(zip_path)
    print(f'✅ ZIP完成: {zip_path.name} ({zip_path.stat().st_size/1024**2:.1f} MiB)')
except Exception as exc:
    print('⚠️ ZIP作成に失敗。MP4とレポートは残っています:',exc)
    if zip_tmp.exists():zip_tmp.unlink()

try:
    display(FileLink(str(target)))
    display(FileLink(str(report_txt)))
    if zip_path.is_file():display(FileLink(str(zip_path)))
except Exception as exc:
    print('ℹ️ ダウンロードリンクの表示を省略:',exc)

try:
    if info['size'] <= 20*1024**2:
        b64=base64.b64encode(target.read_bytes()).decode('ascii')
        display(HTML('<video controls playsinline preload="metadata" width="480" '
                     'style="max-width:100%" src="data:video/mp4;base64,'+b64+'"></video>'))
    else:
        print('ℹ️ 大容量動画なのでプレビューを省略（ファイルには影響なし）。')
except Exception as exc:
    print('⚠️ プレビュー表示のみ失敗。動画・時間レポートは保存済み:',exc)

if globals().get('AUTO_DOWNLOAD',True):
    try:
        if zip_path.is_file():
            files.download(str(zip_path))
            print('✅ MP4＋時間レポートのZIPダウンロードをリクエストしました。')
        else:
            files.download(str(target))
            files.download(str(report_txt))
    except Exception as exc:
        print('⚠️ ブラウザダウンロードを開始できません。左のファイル一覧から取得してください:',exc)
    if globals().get('DOWNLOAD_MP4_SEPARATELY',False):
        try:files.download(str(target))
        except Exception as exc:print('⚠️ MP4単体のダウンロード開始に失敗:',exc)
else:
    print('ℹ️ 自動ダウンロードOFF。上のリンクかColabファイル一覧から取得してください。')
print('📁 保存先:',current_job)
print('⑨を再実行しても動画の再生成は不要です。')


## ⑩ 緊急ダウンロード（必要な場合のみ）

完了後にダウンロードだけ失敗した場合、ZIPを再取得できます。

In [ ]:
# @title ⑩ 緊急再ダウンロード（既存MP4＋生成時間のZIP）
from pathlib import Path
from google.colab import files
from IPython.display import display, FileLink
import zipfile, json
root=Path('/content/infinitetalk_outputs')
completed=sorted(root.glob('*/InfiniteTalk_*s_*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
if not completed:
    print('完成MP4がありません。⑧または⑨が成功しているか確認してください。')
else:
    video=completed[0]
    job=video.parent
    packages=sorted(job.glob('*_MP4_and_Timing.zip'),key=lambda p:p.stat().st_mtime,reverse=True)
    if packages:
        package=packages[0]
    else:
        report=job/'generation_time_report.txt'
        if not report.is_file():
            metrics_path=job/'metrics_latest.json'
            if metrics_path.is_file():
                m=json.loads(metrics_path.read_text(encoding='utf-8'))
                report.write_text('InfiniteTalk 生成時間レポート\n'
                     f'動画: {video.name}\n'
                     f'⑧ モデル読み込み＋推論: {m.get("generation_seconds","記録なし")} 秒\n'
                     f'⑨ 仕上げ: {m.get("finalization_seconds","記録なし")} 秒\n'
                     f'⑧＋⑨ 合計: {m.get("combined_processing_seconds","記録なし")} 秒\n'
                     '※ 初回モデル取得・アップロード・ダウンロード時間は含みません。\n',encoding='utf-8-sig')
            else:
                report.write_text('生成時間の計測記録が見つかりません。動画は保存されています。\n',encoding='utf-8-sig')
        package=job/f'InfiniteTalk_{job.name}_MP4_and_Timing.zip'
        with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
            for p in (video,report,job/'generation_time_report.json',job/'metrics_latest.json',root/'generation_metrics.csv'):
                if p.is_file():z.write(p,arcname=p.name)
        print('✅ ZIPを再作成しました。')
    print('MP4:',video)
    print('ZIP:',package)
    display(FileLink(str(package)))
    try:files.download(str(package))
    except Exception as exc:print('自動ダウンロード不可。上のリンクから保存してください:',exc)


## ⑪ 生成時間の履歴・CSV単体ダウンロード（任意）

In [ ]:
# @title 生成時間を一覧表示・CSVダウンロード
from pathlib import Path
from google.colab import files
import pandas as pd

csv_path = Path('/content/infinitetalk_outputs/generation_metrics.csv')
if not csv_path.is_file():
    print('計測データがありません。⑥を実行すると生成時間が記録されます。')
else:
    df = pd.read_csv(csv_path,encoding='utf-8-sig')
    if df.empty:
        print('計測履歴はまだありません。')
    else:
        keep = ['started_utc','state','gpu','mode','teacache','target_seconds',
                'generation_seconds','finalization_seconds',
                'combined_processing_seconds','completed_chunks']
        print('✅ 生成時間履歴（秒）。同じ音声長で比較してください。')
        display(df[[col for col in keep if col in df.columns]].tail(20))
        for _, r in df[df['state']=='success'].tail(10).iterrows():
            print(f"{r['run_id']} TeaCache={r['teacache']} / "
                  f"音声={r['target_seconds']}秒 / ⑥={float(r['generation_seconds'])/60:.2f}分")
    files.download(str(csv_path))

---
### 重要な注意

- ①のアップロードファイルはColabの一時ストレージに保存されます。ランタイムを切断・リセットすると消える場合があります。生成後はZIPを手元に保存してください
- ②`TEST_FIRST_SECONDS=0`で音声全長、`5`なら最初の5秒だけテスト。③で長さに合わせて生成ジョブを準備します
- ⑥モデルの初回取得は大容量で時間がかかります。同じランタイム・ファイルが残っていれば再ダウンロードを省けます
- **画質や速度はv8から変えていません。** 入力順序とMP4＋生成時間データの取得方法だけを改善しました
- ブラウザのZIPダウンロードが起動しなくても、Colab左側「ファイル」→`/content/infinitetalk_outputs/＜ジョブ名＞/` にMP4・時間レポート・ZIPが残ります
